# SOP: Querying the Endotype Catalog

This notebook demonstrates two common workflows:

1. **Endotype-centered query:** retrieve genes, protein–protein interactions, and phenotypes associated with an endotype.
2. **Gene-centered query:** retrieve associated endotypes and endotype-specific interaction partners for a gene.


## 1. Import libraries

In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)


## 2. Download and connect to the Endotype Catalog

The Endotype Catalog is stored as a SQLite database (`endotype_catalog.db`) in the project GitHub repository and the Repo4eu website. The code below automatically downloads the database if it is not already available in the working directory.


In [ ]:
import sqlite3
from pathlib import Path
from urllib.request import urlretrieve

DB_FILE = Path("endotype_catalog.db")
DB_URL = (
    "https://raw.githubusercontent.com/"
    "bwh784/EndotypeCatalog/main/endotype_catalog.db"
)  ## this could also be the Repo4eu website

# Download the database only if it is not already available locally
if not DB_FILE.exists():
    print("Downloading Endotype Catalog database...")
    urlretrieve(DB_URL, DB_FILE)
    print("Download complete.")
else:
    print("Using existing local database.")

# Open the database in read-only mode
db_path = DB_FILE.resolve()
conn = sqlite3.connect(f"file:{db_path.as_posix()}?mode=ro", uri=True)
print("Connected to:", db_path)


## 3. Inspect available tables

In [ ]:
pd.read_sql_query(
    "SELECT name AS table_name FROM sqlite_master WHERE type='table' ORDER BY name",
    conn
)


## 4. Browse available endotypes

In [ ]:
available_endotypes = pd.read_sql_query(
    '''
    SELECT endotype_id, endotype AS endotype_name
    FROM endotypes
    ORDER BY endotype_id
    ''', conn
)
available_endotypes


# Part I. Endotype-centered query

Choose an endotype ID. `R4EU:000001` is used as an example.


In [ ]:
ENDOTYPE_ID = "R4EU:000001"

## 5. Endotype information

In [ ]:
endotype_info = pd.read_sql_query(
    '''
    SELECT endotype_id, endotype AS endotype_name
    FROM endotypes
    WHERE endotype_id = ?
    ''', conn, params=(ENDOTYPE_ID,)
)
endotype_info


## 6. Associated genes

In [ ]:
endotype_genes = pd.read_sql_query(
    '''
    SELECT DISTINCT ge.gene_symbol, g.entrez_id
    FROM gene_endotype ge
    LEFT JOIN genes g ON ge.gene_symbol = g.gene_symbol
    WHERE ge.endotype_id = ?
    ORDER BY ge.gene_symbol
    ''', conn, params=(ENDOTYPE_ID,)
)
print(f"Number of associated genes: {len(endotype_genes):,}")
endotype_genes


## 7. Endotype-specific protein–protein interactions

In [ ]:
endotype_interactions = pd.read_sql_query(
    '''
    SELECT DISTINCT interactor1, interactor2
    FROM edges
    WHERE endotype_id = ?
    ORDER BY interactor1, interactor2
    ''', conn, params=(ENDOTYPE_ID,)
)
print(f"Number of interactions: {len(endotype_interactions):,}")
endotype_interactions


## 8. Associated phenotypes

In [ ]:
endotype_phenotypes = pd.read_sql_query(
    '''
    SELECT DISTINCT phenotype
    FROM phenotypes
    WHERE endotype_id = ?
    ORDER BY phenotype
    ''', conn, params=(ENDOTYPE_ID,)
)
print(f"Number of phenotypes: {len(endotype_phenotypes):,}")
endotype_phenotypes


## 9. Endotype summary

In [ ]:
endotype_name = endotype_info.iloc[0]["endotype_name"] if not endotype_info.empty else None

pd.DataFrame({
    "endotype_id": [ENDOTYPE_ID],
    "endotype_name": [endotype_name],
    "genes": [len(endotype_genes)],
    "protein_protein_interactions": [len(endotype_interactions)],
    "phenotypes": [len(endotype_phenotypes)]
})


# Part II. Gene-centered query

Choose a gene symbol. `ACVR1` is used as an example.


In [ ]:
GENE_SYMBOL = "ACVR1"

## 10. Gene information

In [ ]:
gene_info = pd.read_sql_query(
    "SELECT gene_symbol, entrez_id FROM genes WHERE gene_symbol = ?",
    conn, params=(GENE_SYMBOL,)
)
gene_info


## 11. Endotypes associated with the gene

In [ ]:
gene_endotypes = pd.read_sql_query(
    '''
    SELECT DISTINCT ge.endotype_id, e.endotype AS endotype_name
    FROM gene_endotype ge
    LEFT JOIN endotypes e ON ge.endotype_id = e.endotype_id
    WHERE ge.gene_symbol = ?
    ORDER BY ge.endotype_id
    ''', conn, params=(GENE_SYMBOL,)
)
print(f"Number of associated endotypes: {len(gene_endotypes):,}")
gene_endotypes


## 12. Interaction partners for the gene

Protein–protein interactions are treated as **undirected**. A gene can therefore occur as either `interactor1` or `interactor2`. The query searches both columns and reports the other gene as the interaction partner while retaining the endotype context.


In [ ]:
gene_interactions = pd.read_sql_query(
    '''
    SELECT DISTINCT
        CASE WHEN ed.interactor1 = ? THEN ed.interactor2
             ELSE ed.interactor1 END AS interaction_partner,
        ed.endotype_id,
        en.endotype AS endotype_name
    FROM edges ed
    LEFT JOIN endotypes en ON ed.endotype_id = en.endotype_id
    WHERE ed.interactor1 = ? OR ed.interactor2 = ?
    ORDER BY ed.endotype_id, interaction_partner
    ''', conn, params=(GENE_SYMBOL, GENE_SYMBOL, GENE_SYMBOL)
)
print(f"Number of endotype-specific interaction records: {len(gene_interactions):,}")
gene_interactions


## 13. Gene summary

In [ ]:
pd.DataFrame({
    "gene_symbol": [GENE_SYMBOL],
    "associated_endotypes": [len(gene_endotypes)],
    "endotype_specific_interaction_records": [len(gene_interactions)],
    "unique_interaction_partners": [
        gene_interactions["interaction_partner"].nunique() if not gene_interactions.empty else 0
    ]
})


# Part III. Reusable query functions

The following functions allow the same workflow to be applied to any endotype or gene.


In [ ]:
def query_endotype(endotype_id, conn=conn):
    info = pd.read_sql_query(
        "SELECT endotype_id, endotype AS endotype_name FROM endotypes WHERE endotype_id=?",
        conn, params=(endotype_id,))
    genes = pd.read_sql_query(
        '''SELECT DISTINCT ge.gene_symbol, g.entrez_id
           FROM gene_endotype ge LEFT JOIN genes g ON ge.gene_symbol=g.gene_symbol
           WHERE ge.endotype_id=? ORDER BY ge.gene_symbol''',
        conn, params=(endotype_id,))
    interactions = pd.read_sql_query(
        "SELECT DISTINCT interactor1, interactor2 FROM edges WHERE endotype_id=? ORDER BY interactor1, interactor2",
        conn, params=(endotype_id,))
    phenotypes = pd.read_sql_query(
        "SELECT DISTINCT phenotype FROM phenotypes WHERE endotype_id=? ORDER BY phenotype",
        conn, params=(endotype_id,))
    return {"endotype":info, "genes":genes, "interactions":interactions, "phenotypes":phenotypes}


def query_gene(gene_symbol, conn=conn):
    gene = pd.read_sql_query(
        "SELECT gene_symbol, entrez_id FROM genes WHERE gene_symbol=?",
        conn, params=(gene_symbol,))
    endotypes = pd.read_sql_query(
        '''SELECT DISTINCT ge.endotype_id, e.endotype AS endotype_name
           FROM gene_endotype ge LEFT JOIN endotypes e ON ge.endotype_id=e.endotype_id
           WHERE ge.gene_symbol=? ORDER BY ge.endotype_id''',
        conn, params=(gene_symbol,))
    interactions = pd.read_sql_query(
        '''SELECT DISTINCT
              CASE WHEN ed.interactor1=? THEN ed.interactor2 ELSE ed.interactor1 END AS interaction_partner,
              ed.endotype_id, en.endotype AS endotype_name
           FROM edges ed LEFT JOIN endotypes en ON ed.endotype_id=en.endotype_id
           WHERE ed.interactor1=? OR ed.interactor2=?
           ORDER BY ed.endotype_id, interaction_partner''',
        conn, params=(gene_symbol, gene_symbol, gene_symbol))
    return {"gene":gene, "endotypes":endotypes, "interactions":interactions}


## 14. Example: reusable endotype query

In [ ]:
result = query_endotype("R4EU:000001")
display(result["endotype"])
display(result["genes"].head(10))
display(result["interactions"].head(10))
display(result["phenotypes"])


## 15. Example: reusable gene query

In [ ]:
result = query_gene("ACVR1")
display(result["gene"])
display(result["endotypes"])
display(result["interactions"].head(20))


## 16. Optional export

In [ ]:
# Uncomment as needed:
# endotype_genes.to_csv(f"{ENDOTYPE_ID}_genes.csv", index=False)
# endotype_interactions.to_csv(f"{ENDOTYPE_ID}_interactions.csv", index=False)
# endotype_phenotypes.to_csv(f"{ENDOTYPE_ID}_phenotypes.csv", index=False)
# gene_endotypes.to_csv(f"{GENE_SYMBOL}_endotypes.csv", index=False)
# gene_interactions.to_csv(f"{GENE_SYMBOL}_interactions.csv", index=False)


## 17. Close the database connection

In [ ]:
conn.close()
print("Database connection closed.")


## Interpretation notes

- An endotype query retrieves its molecular module and associated phenotypic annotations.
- A gene query identifies the endotypes containing that gene and its interaction partners in those endotype-specific networks.
- Both interaction columns are searched because protein–protein interactions are treated as undirected.
- The same gene or interaction may occur in multiple endotypes; retaining `endotype_id` preserves this context.
